In [1]:
from pathlib import Path  # Dùng để tìm thư mục dự án
import sys  # Dùng để Python tìm thấy các module trong thư mục src/.

project_root = Path.cwd()  # Lấy thư mục hiện tại của kernel notebook.
if project_root.name == 'notebooks':  # Nếu kernel đang ở notebooks/, lùi lên một cấp.
    project_root = project_root.parent  # Thư mục cha chính là gốc repository.
sys.path.insert(0, str(project_root))  # Thêm gốc dự án vào danh sách nơi Python tìm module.
print('Thư mục dự án:', project_root)  # In ra để người học xác nhận đúng workspace.


Thư mục dự án: c:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project


## 3. Kiểm tra dữ liệu và GPU trước khi chạy

Hai JSONL cần được tạo từ notebook EDA. Cần dùng môi trường Python đã cài các thư viện trong `requirements.txt`, đặc biệt là bản PyTorch có CUDA. Nếu không thấy GPU ở cell dưới, đừng chạy fine-tune vì CPU sẽ chậm hơn rất nhiều.


In [2]:
import torch  # PyTorch kiểm tra GPU và thực hiện phép tính của mô hình.
from src.train_qa import TRAIN_FILE, DEV_FILE  # Lấy đường dẫn dữ liệu mà script sẽ dùng.

print('Train tồn tại:', TRAIN_FILE.exists(), TRAIN_FILE)  # Xác nhận tệp train đã sẵn sàng.
print('Dev tồn tại:', DEV_FILE.exists(), DEV_FILE)  # Xác nhận tệp dev đã sẵn sàng.
print('Có CUDA:', torch.cuda.is_available())  # CUDA cho phép PyTorch dùng GPU NVIDIA.
if torch.cuda.is_available():  # Chỉ hỏi tên thiết bị khi đã phát hiện GPU.
    print('GPU:', torch.cuda.get_device_name(0))  # Kiểm tra notebook nhận đúng card
    print('Hỗ trợ BF16:', torch.cuda.is_bf16_supported())  # BF16 thường tiết kiệm bộ nhớ và ổn định.


c:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train tồn tại: True C:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\data\processed\train.json
Dev tồn tại: True C:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\data\processed\dev.json
Có CUDA: True
GPU: NVIDIA GeForce RTX 5070 Ti
Hỗ trợ BF16: True


## 4. Chạy thí nghiệm

Cell sau gọi cùng một pipeline dùng cho hai kiến trúc. Với mỗi seed, script tạo mô hình mới từ checkpoint gốc, token hóa dữ liệu, fine-tune ba epoch, đánh giá EM/F1 sau từng epoch và lưu checkpoint tốt nhất theo F1. Không tái sử dụng trọng số của seed trước.

Mỗi lượt sẽ ghi `metrics.json`, `predictions.json` và thư mục `best_model/` dưới `results/finetune/runs/run-.../`. Xem đầy đủ logic từng dòng trong `src/train_qa.py`; đặc biệt là `prepare_train_features`, `decode_predictions` và `make_compute_metrics`.

Để xem phiên bản dữ liệu, revision model và đường dẫn trước khi train, đặt `plan_only=True` ở cell bên dưới. Đặt lại `False` khi muốn chạy thật.


In [3]:
from src.train_qa import run_experiment  # Nạp hàm điều phối tiền xử lý, fine-tune và đánh giá.

seeds = [42, 123, 1024]  # Ba seed theo quy ước của project để đo độ biến động.
model_names = ['xlm-roberta']  # Notebook này chỉ chạy kiến trúc xlm-roberta-base.
# True chỉ xem kế hoạch; False mới fine-tune.
plan_only = False
experiment_dir = run_experiment(model_names, seeds, plan_only=plan_only)


Phiên bản dữ liệu: uit-viquad2-05de3ab29680-ae347fbe9385-0bfbe5821468
Thư mục thí nghiệm: C:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\results\finetune\runs\run-fe5f5f5b27578bd4
Seed yêu cầu: [42, 123, 1024]
Batch hiệu dụng trên mỗi GPU: 4 x 8 = 32
xlm-roberta: revision e73636d4f797dec63c3081bb6ed5c7b0bb3f2089

Model: xlm-roberta-base | seed: 42
GPU: NVIDIA GeForce RTX 5070 Ti
Thư mục seed: C:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\results\finetune\runs\run-fe5f5f5b27578bd4\xlm-roberta\seed-42

--- Tóm tắt dữ liệu trước khi huấn luyện ---
Số mẫu train gốc: 8,000
Số mẫu dev gốc: 3,814
Số train features sau tiền xử lý: 8,474
Số dev features sau tiền xử lý: 3,923

Cấu hình huấn luyện:
Batch train trên mỗi GPU: 4
Số bước tích lũy gradient: 8
Batch hiệu dụng trên mỗi GPU: 4 x 8 = 32
Batch đánh giá trên mỗi GPU: 8
Số epoch: 3
Learning rate: 2e-05
Dùng BF16: True
Dùng F

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2402.56it/s]
[transformers] XLMRobertaForQuestionAnswering LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
qa_outputs.bias             | MISSING    | 
qa_outputs.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Exact Match,F1
1,2.400223,2.060149,33.953854,43.653858
2,1.739448,1.801125,40.456214,49.737356
3,1.536993,1.789692,41.452543,51.262564


Writing model shards: 100%|██████████| 1/1 [00:19<00:00, 19.08s/it]


Training Loss,Validation Loss,Epoch,Exact Match,F1
1.536993,1.789692,3,41.452543,51.262564


Kết quả seed 42: EM=41.45, F1=51.26

Model: xlm-roberta-base | seed: 123
GPU: NVIDIA GeForce RTX 5070 Ti
Thư mục seed: C:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\results\finetune\runs\run-fe5f5f5b27578bd4\xlm-roberta\seed-123

--- Tóm tắt dữ liệu trước khi huấn luyện ---
Số mẫu train gốc: 8,000
Số mẫu dev gốc: 3,814
Số train features sau tiền xử lý: 8,474
Số dev features sau tiền xử lý: 3,923

Cấu hình huấn luyện:
Batch train trên mỗi GPU: 4
Số bước tích lũy gradient: 8
Batch hiệu dụng trên mỗi GPU: 4 x 8 = 32
Batch đánh giá trên mỗi GPU: 8
Số epoch: 3
Learning rate: 2e-05
Dùng BF16: True
Dùng FP16: False

Ví dụ nhãn của một train feature:
Feature index: 6
Start position: 55
End position: 67
Token được gán nhãn: ['▁hoạt', '▁động', '▁văn', '▁chương', ',', '▁âm', '▁nhạc', ',', '▁nghệ', '▁thuật', '▁và', '▁sân', '▁khấu']
Đoạn giải mã từ nhãn: hoạt động văn chương, âm nhạc, nghệ thuật và sân khấu


... (6 outputs left out)

Epoch,Training Loss,Validation Loss,Exact Match,F1
1,2.896935,2.344867,28.369166,37.172816
2,1.978801,1.961393,38.463555,45.281521
3,1.705885,1.897271,39.564761,48.348924


Writing model shards: 100%|██████████| 1/1 [00:25<00:00, 25.35s/it]


Training Loss,Validation Loss,Epoch,Exact Match,F1
1.705885,1.897271,3,39.564761,48.348924


Kết quả seed 1024: EM=39.56, F1=48.35

Tổng hợp (F1/EM theo phần trăm):
xlm-roberta: seed tốt nhất=42, F1=51.26, EM=41.45; F1 trung bình=50.21 ± 1.32
Đã lưu tổng hợp tại: C:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\results\finetune\runs\run-fe5f5f5b27578bd4\summary-seeds-42-123-1024.json


## 5. Đọc kết quả

`summary.json` ghi seed có F1 cao nhất và điểm trung bình ± độ lệch chuẩn của các seed trong lần chạy. `predictions.json` lưu câu trả lời từng mẫu để xem lỗi. Tên `best_seed` chỉ có nghĩa là seed tốt nhất trong ba seed đã chạy trên tập dev này; nó không đảm bảo sẽ tốt nhất trên tập test hoặc dữ liệu mới.

Khi so sánh XLM-RoBERTa với mDeBERTa, so sánh cả F1/EM trung bình và độ lệch chuẩn, dùng cùng train/dev, seed, epoch, độ dài token và ngưỡng no-answer. Sau khi chọn cấu hình, chỉ dùng test set một lần để báo cáo cuối cùng nếu dự án có test set riêng.


In [4]:
# Tên summary gắn với đúng danh sách seed của lượt vừa chạy.
seed_suffix = "-".join(str(seed) for seed in sorted(set(seeds)))
summary_path = experiment_dir / model_names[0] / f"summary-seeds-{seed_suffix}.json"
if plan_only:
    print("Đang xem kế hoạch. Đặt plan_only=False khi muốn huấn luyện.")
elif summary_path.exists():
    print(summary_path.read_text(encoding="utf-8"))
else:
    print("Chưa có summary ở thư mục thí nghiệm:", summary_path)


{
  "experiment_id": "run-fe5f5f5b27578bd4",
  "dataset_version": "uit-viquad2-05de3ab29680-ae347fbe9385-0bfbe5821468",
  "requested_seeds": [
    42,
    123,
    1024
  ],
  "best_seed": 42,
  "best_f1": 51.262564034541036,
  "best_exact_match": 41.45254326166754,
  "mean_f1": 50.21455651819528,
  "std_f1": 1.3225496823350096,
  "mean_exact_match": 40.56109071840587,
  "std_exact_match": 0.7742434745796251
}
